# Node2Vec Phase 0 - Undirected Structural Branch

This notebook prepares the new structural branch for the next report.

It does four things:

1. builds an undirected train/test split for structural recommendation
2. fits a new power-law correction parameter on the undirected train graph
3. trains Node2Vec on the undirected train graph
4. saves both uncorrected and degree-corrected Node2Vec caches for later evaluation

The later comparison notebook (`eval-3.ipynb`) should load these caches rather than recomputing the structural branch from scratch.


In [ ]:
import os
import sys
import json
import math
import random
import pickle
import warnings
from collections import Counter

import numpy as np
import pandas as pd
import networkx as nx
import powerlaw

warnings.filterwarnings('ignore')

REPO_ROOT = r'C:\\programming\\github-repos\\graph-ending'
CACHE_DIR = os.path.join(REPO_ROOT, 'WikiCS', 'custom-wiki', 'cache', 'node2vec-2')
os.makedirs(CACHE_DIR, exist_ok=True)

DATA_PATH = '../../data/data-embeddings.json'

utils_parent = os.path.join(REPO_ROOT, 'WikiCS', 'custom-wiki')
sys.path.insert(0, utils_parent)

from utils import load_graph_data, node2vec_svd

SEED = 42
TRAIN_RATIO = 0.8
TEST_RATIO = 0.2
N2V_DIM = 128
N2V_NUM_WALKS = 10
N2V_WALK_LENGTH = 80

print('Setup done.')
print(f'Cache dir: {CACHE_DIR}')


## Step 1 - Load Full Graph

The raw WikiCS graph is directed, but this branch converts it to an undirected structural graph before splitting because the target use case is article relatedness and Node2Vec itself operates on undirected structure here.


In [ ]:
nodes, id_to_title, title_to_id, G_full = load_graph_data(DATA_PATH)
G_full_und = G_full.to_undirected()

print(f'Full graph: {G_full.number_of_nodes()} nodes, {G_full.number_of_edges()} directed edges')
print(f'Undirected structural graph: {G_full_und.number_of_nodes()} nodes, {G_full_und.number_of_edges()} edges')


## Step 2 - Build Undirected Train/Test Split

This branch treats structural relatedness as an undirected relation.

Procedure:

1. convert the full graph to an undirected graph
2. shuffle the undirected edges
3. greedily assign about 20% of edges to test
4. never remove a node's final remaining train edge
5. repair any accidental new train isolates by moving one held-out edge back to train
6. keep the remaining edges as the structural train graph

This keeps the split recommendation-oriented while avoiding the earlier reverse-direction leakage issue and also avoiding unnecessary train-time isolates.


In [ ]:
random.seed(SEED)
np.random.seed(SEED)

undirected_edges = list(G_full_und.edges())
random.shuffle(undirected_edges)

n_edges_und = len(undirected_edges)
target_test = int(np.ceil(TEST_RATIO * n_edges_und))

remaining_degree = Counter(dict(G_full_und.degree()))
E_test = []
E_train = []
blocked_for_train_preservation = 0

for u, v in undirected_edges:
    can_hold_out = (
        len(E_test) < target_test and
        remaining_degree[u] > 1 and
        remaining_degree[v] > 1
    )
    if can_hold_out:
        E_test.append((u, v))
        remaining_degree[u] -= 1
        remaining_degree[v] -= 1
    else:
        E_train.append((u, v))
        if len(E_test) < target_test and not (remaining_degree[u] > 1 and remaining_degree[v] > 1):
            blocked_for_train_preservation += 1

G_train_struct = nx.Graph()
G_train_struct.add_nodes_from(G_full_und.nodes())
G_train_struct.add_edges_from(E_train)

original_isolates = {n for n in G_full_und.nodes() if G_full_und.degree(n) == 0}
new_isolates = [
    n for n in G_train_struct.nodes()
    if G_train_struct.degree(n) == 0 and n not in original_isolates
]

repaired_isolates = 0
if new_isolates:
    test_edge_set = set(E_test)
    for nid in new_isolates:
        for nbr in G_full_und.neighbors(nid):
            edge = (nid, nbr) if (nid, nbr) in test_edge_set else (nbr, nid)
            if edge in test_edge_set:
                test_edge_set.remove(edge)
                E_test.remove(edge)
                E_train.append(edge)
                G_train_struct.add_edge(*edge)
                repaired_isolates += 1
                break

eval_nodes = {
    node
    for edge in E_test
    for node in edge
}

isolated_after_split = [n for n in G_train_struct.nodes() if G_train_struct.degree(n) == 0]

print(f'Undirected edges total     : {n_edges_und}')
print(f'Target undirected test     : {target_test}')
print(f'Actual undirected test     : {len(E_test)}')
print(f'Undirected train edges     : {len(E_train)}')
print(f'Train graph nodes          : {G_train_struct.number_of_nodes()}')
print(f'Train graph edges          : {G_train_struct.number_of_edges()}')
print(f'Eval nodes                 : {len(eval_nodes)}')
print(f'Blocked to preserve train  : {blocked_for_train_preservation}')
print(f'Repaired new isolates      : {repaired_isolates}')
print(f'Isolated train nodes       : {len(isolated_after_split)}')
print(f'Original structural isolates: {len(original_isolates)}')

SPLIT_FILE = os.path.join(CACHE_DIR, 'undirected_split.pkl')
with open(SPLIT_FILE, 'wb') as f:
    pickle.dump({
        'E_train': E_train,
        'E_test': E_test,
        'G_train_struct': G_train_struct,
        'eval_nodes': eval_nodes,
        'isolated_train_nodes': isolated_after_split,
        'original_isolates': sorted(original_isolates),
        'seed': SEED,
        'train_ratio': TRAIN_RATIO,
        'test_ratio': TEST_RATIO,
        'target_test_edges': target_test,
        'actual_test_edges': len(E_test),
        'blocked_for_train_preservation': blocked_for_train_preservation,
        'repaired_new_isolates': repaired_isolates,
    }, f)
print(f'Saved to {SPLIT_FILE}')


## Step 3 - Fit Power-Law Alpha on the Undirected Train Graph

The corrected Node2Vec branch should fit its alpha on the same undirected train graph used for structural learning.


In [ ]:
train_degrees_und = np.array([max(G_train_struct.degree(n), 1) for n in G_train_struct.nodes()], dtype=float)
fit_train = powerlaw.Fit(train_degrees_und, discrete=True, verbose=False)
alpha_train = fit_train.power_law.alpha
xmin_train = fit_train.power_law.xmin

print('Undirected train graph power-law fit:')
print(f'  alpha = {alpha_train:.4f}')
print(f'  xmin  = {xmin_train}')
print(f'  sigma = {fit_train.power_law.sigma:.4f}')

ALPHA_FILE = os.path.join(CACHE_DIR, 'alpha_structural.pkl')
with open(ALPHA_FILE, 'wb') as f:
    pickle.dump({
        'alpha': alpha_train,
        'xmin': xmin_train,
        'degrees_undirected': train_degrees_und,
    }, f)
print(f'Saved to {ALPHA_FILE}')


## Step 4 - Train Node2Vec on the Undirected Train Graph

This uses the same SVD-based pure-numpy Node2Vec pipeline already used elsewhere in the repo, but now on the corrected undirected structural branch.


In [ ]:
print('Training Node2Vec on undirected train graph (SVD-based)...')
n2v_emb = node2vec_svd(
    G_train_struct,
    num_walks=N2V_NUM_WALKS,
    walk_length=N2V_WALK_LENGTH,
    embedding_dim=N2V_DIM,
    seed=SEED,
)

n2v_ids = sorted(n2v_emb.keys())
df_n2v_uncorrected = pd.DataFrame({
    'id': n2v_ids,
    'embedding': [n2v_emb[nid] for nid in n2v_ids],
})

N2V_UNCORR_FILE = os.path.join(CACHE_DIR, 'node2vec_undirected_uncorrected.parquet')
df_n2v_uncorrected.to_parquet(N2V_UNCORR_FILE, index=False)
print(f'Saved to {N2V_UNCORR_FILE}')
print(f'Embeddings: {len(df_n2v_uncorrected)} nodes, dim={len(df_n2v_uncorrected["embedding"].iloc[0])}')


## Step 5 - Build Degree-Corrected Node2Vec Cache

This cache applies the same popularity-damping idea to the structural embeddings, but now fitted on the undirected train graph.


In [ ]:
degree_und = {nid: max(G_train_struct.degree(nid), 1) for nid in G_train_struct.nodes()}

corrected_rows = []
for _, row in df_n2v_uncorrected.iterrows():
    nid = row['id']
    emb = np.asarray(row['embedding'])
    norm = math.pow(np.log(degree_und[nid] + 1.0), alpha_train)
    corrected_rows.append({
        'id': nid,
        'embedding': emb / norm,
        'degree_undirected': degree_und[nid],
        'norm': norm,
    })

df_n2v_corrected = pd.DataFrame(corrected_rows)
N2V_CORR_FILE = os.path.join(CACHE_DIR, 'node2vec_undirected_corrected.parquet')
df_n2v_corrected.to_parquet(N2V_CORR_FILE, index=False)
print(f'Saved to {N2V_CORR_FILE}')


## Step 6 - Save Branch Metadata

This metadata file is meant to make `eval-3.ipynb` simpler and less error-prone.


In [ ]:
metadata = {
    'branch_name': 'node2vec-2 undirected structural branch',
    'seed': SEED,
    'split_rule': 'convert full graph to undirected, greedily hold out about 20 percent of undirected edges, avoid removing a node\'s final train edge, and repair any accidental new train isolates',
    'n_nodes_full': int(G_full.number_of_nodes()),
    'n_edges_full_directed': int(G_full.number_of_edges()),
    'n_edges_full_undirected': int(G_full_und.number_of_edges()),
    'n_train_edges_undirected': int(len(E_train)),
    'n_test_edges_undirected_target': int(target_test),
    'n_test_edges_undirected_actual': int(len(E_test)),
    'n_eval_nodes': int(len(eval_nodes)),
    'isolated_train_nodes': int(len(isolated_after_split)),
    'original_isolates': int(len(original_isolates)),
    'blocked_for_train_preservation': int(blocked_for_train_preservation),
    'repaired_new_isolates': int(repaired_isolates),
    'node2vec_dim': N2V_DIM,
    'node2vec_num_walks': N2V_NUM_WALKS,
    'node2vec_walk_length': N2V_WALK_LENGTH,
    'alpha_structural': float(alpha_train),
    'artifacts': {
        'split': SPLIT_FILE,
        'alpha': ALPHA_FILE,
        'node2vec_uncorrected': N2V_UNCORR_FILE,
        'node2vec_corrected': N2V_CORR_FILE,
    },
}

META_FILE = os.path.join(CACHE_DIR, 'phase0_metadata.json')
with open(META_FILE, 'w', encoding='utf-8') as f:
    json.dump(metadata, f, indent=2)
print(f'Saved to {META_FILE}')
metadata
